In [ ]:
EPOCHS = None; learnRate = None                                                 # epochN = 300, learnRate = 0.0005/0.0010/0.0015/0.0020/0.0025
import glob
import zipfile
fileList = glob.glob('*.zip')
zip_ref = zipfile.ZipFile(fileList[0], 'r');
zip_ref.extractall()
zip_ref.close()
w = fileList[0].split('.'); project = w[-2]
print('프로젝트:',project)


In [ ]:
import os
import sys
import time
import glob
import random
import pickle
import numpy as np

# TensorFlow / KERAS
import tensorflow as tf
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Input
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Imaging
import cv2 as cv
import matplotlib.pyplot as plt

print('OpenCV Version:       ', cv.__version__)
print('NumPy Version:        ', np.__version__)
print('Tensorflow Version:   ', tf.__version__)
print('Keras Version:        ', tensorflow.keras.__version__)
print('Python Version:       ', sys.version)


In [ ]:
# 이미지 데이터 조향각 추출

with open(f'{project}/_{project}.pickle', 'rb') as f:                           # pickle 파일
    x_exam_Image = pickle.load(f)                                               # 시험(Test) 이미지 파일 이름 리스트
    x_valid_Image = pickle.load(f)                                              # 평가(Valid) 이미지 파일 이름 리스트
    x_train_Image = pickle.load(f)                                              # 훈련(Train) 이미지 파일 이름 리스트
    p = pickle.load(f)                                                          # 파라메터 리스트
if EPOCHS == None: EPOCHS = p[0]                                                # 학습회수
if learnRate == None: learnRate = p[1]                                          # 학습 비율 0.0005/0.0010/0.0015/0.0020/0.0025
#-------------------------------------------------------------------------------
y_exam_Angle=[]
y_valid_Angle=[]
y_train_Angle = []
for f in x_exam_Image:                                                          # 시험용 이미지 파일 이름에서 각도값
    y_exam_Angle.append(int(f[-7:-4]))
for f in x_valid_Image:                                                         # 평가용 이미지 파일 이름에서 각도값
    y_valid_Angle.append(int(f[-7:-4]))
for f in x_train_Image:                                                         # 훈련용 이미지 파일 이름에서 각도값
    y_train_Angle.append(int(f[-7:-4]))

print('학습 회수:',EPOCHS,)
print('학습 비율', learnRate)
print('시험 이미지 개수:',len(x_exam_Image))
print('평가 이미지 개수:',len(x_valid_Image))
print('훈련 이미지 개수:',len(x_train_Image))


In [4]:
# Nvidia CNN 모델 구성 -----------------------------------------------------
def nvidia_model(lr=learnRate):

    model = Sequential(name='Nvidia_Model')

    # elu=Expenential Linear Unit, similar to leaky Relu
    # skipping 1st hiddel layer (nomralization layer), as we have normalized the data
    # Convolution Layers

    model.add(Input(shape=(66, 200, 3)))
    model.add(Conv2D(24, (5, 5), strides=(2, 2), activation='elu'))
    model.add(Conv2D(36, (5, 5), strides=(2, 2), activation='elu'))
    model.add(Conv2D(48, (5, 5), strides=(2, 2), activation='elu'))
    model.add(Conv2D(64, (3, 3), activation='elu'))
    #model.add(Dropout(0.2))
    model.add(Conv2D(64, (3, 3), activation='elu'))

    # Fully Connected Layers
    model.add(Flatten())
    model.add(Dropout(0.2))
    model.add(Dense(100, activation='elu'))
    model.add(Dense(50, activation='elu'))
    model.add(Dense(10, activation='elu'))

    # output layer: turn angle (from -99 ~ +99  0 is straight, -99 turn left, +99 turn right)
    model.add(Dense(1))

    # since this is a regression problem not classification problem,
    # we use MSE (Mean Squared Error) as loss function
    # optimizer = Adam(lr=1e-3)  # lr is learning rate
    optimizer = Adam(learning_rate=lr)  # 학습율 설정
    model.compile(loss='mse', optimizer=optimizer)

    return model

# 학습 데이터 생성 -----------------------------------------------------------
def batchGen(imageList, angleList, batchSize):

    while True:
        imageBatch = []
        angleBatch = []

        for i in range(batchSize):
            t = f'{project}/'+ imageList[i]
            image = cv.imread(t)
            angle = angleList[i]

            image = image/255 # 255 로 나누어 0 - 0.999 로 데이터 표시 방법 변경
            imageBatch.append(image)
            angleBatch.append(angle)

        yield(np.asarray(imageBatch), np.asarray(angleBatch))


In [5]:
# 모델 학습 ----------------------------------------------
def learnProc():

    startLearn = time.time()   #  학습 시작 시각
    cPCallback = tensorflow.keras.callbacks.ModelCheckpoint(filepath=f'./_{project}_model_check.keras', verbose=1, save_best_only=True)
    hist = model.fit(batchGen(x_train_Image, y_train_Angle, batchSize=len(x_train_Image)),
                              steps_per_epoch=100,
                              epochs=3,                # 학습 회수이며 디폴트로 3 회
                              validation_data = batchGen(x_valid_Image, y_valid_Angle, batchSize=len(x_valid_Image)),
                              validation_steps=200,
                              verbose=1,                    # 학습 시간을 오래 기다려야 하므로 막대 그래프로 진행상황 표시
                              shuffle=True,                    # 주어진 데이터를 랜덤하게 재배치.
                              callbacks=[cPCallback])

    # TensorFlow Lite 변환기 생성
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # 양자화 (선택 사항)
    # converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # 변환 및 저장
    tflite_model = converter.convert()

    # TensorFlow Lite 모델 저장
    open(f'./_{project}_model_final.tflite', 'wb').write(tflite_model)
    model.save(f'./_{project}_model_final.keras')   # 학습모델 저장

    endLearn = time.time()                   # 학습 종료 시각
    timeLearn = int(endLearn - startLearn)   # 학습 소요 시간

    t = f'{int(timeLearn/60):4d} 분   {int(timeLearn%60):2d} 초'
    print('학습 소요시간:',t)

    return(hist.history)   # 딕셔너리 데이터로 변환하여 반환


In [ ]:
model = nvidia_model()              # NVIDIA 모델
model.summary()
history = learnProc()               # 학습 시작

plt.plot(history['loss'], color='blue')
plt.plot(history['val_loss'],color='red')
plt.legend(['training loss','validation loss'])
plt.title(f'Deep Learn - {project}')
plt.savefig(f'./_{project}_Plt.png')
plt.show()
